<a href="https://colab.research.google.com/github/Miranics/TradesREvamp/blob/main/research/05_realtime_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Real-time check before the Expert Advisor

The earlier tests could let an order start a few minutes before the break that triggers it was confirmed. A bot cannot do that. This notebook re-runs the sweep model with the timing a bot can actually follow, checks that the edge survives, and saves the trade list the bot will be compared against in MT5's Strategy Tester.

**Step 1.** Connect Google Drive (approve the pop-up).

In [1]:
import glob
import os

from google.colab import drive

drive.mount("/content/drive")

matches = [p for p in glob.glob("/content/drive/MyDrive/*") if os.path.basename(p).strip() == "My Trades"]
if not matches:
    raise FileNotFoundError("Could not find a folder called 'My Trades' at the top of My Drive")
BASE = matches[0]
DATA = os.path.join(BASE, "data")
OUT = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("Data folder:   ", DATA)
print("Results folder:", OUT)


Mounted at /content/drive
Data folder:    /content/drive/MyDrive/My Trades /data
Results folder: /content/drive/MyDrive/My Trades /results


**Step 2.** Load the Exness gold data saved by the bake-off.

In [2]:
import glob
import os

import numpy as np
import pandas as pd

SPREAD = 0.30
SPREAD_HIGH = 0.60
OZ_PER_001_LOT = 1


def load_m1(pattern):
    files = sorted(glob.glob(os.path.join(DATA, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {DATA}")
    df = pd.concat(pd.read_csv(f) for f in files)
    df["time"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.drop_duplicates("time").set_index("time").sort_index()
    keep = [c for c in ("open", "high", "low", "close", "spread") if c in df.columns]
    df = df[keep].astype(float)
    return df[df.index.dayofweek < 5]


m1 = load_m1("xauusd_*.csv")
m1_ny = m1.tz_convert("America/New_York")

if "spread" in m1.columns:
    ny_spread = m1.loc[(m1.index.hour >= 13) & (m1.index.hour < 21), "spread"].median()
    SPREAD = round(float(ny_spread), 2)
    SPREAD_HIGH = round(2 * SPREAD, 2)
print(f"Cost per position: ${SPREAD:.2f} (Exness median spread in New York hours); stress test: ${SPREAD_HIGH:.2f}")

per_month = m1.groupby(m1.index.strftime("%Y-%m")).size()
print(f"Gold minutes loaded: {len(m1):,}  from {m1.index[0]:%Y-%m-%d} to {m1.index[-1]:%Y-%m-%d}")
thin = per_month.iloc[:-1][per_month.iloc[:-1] < 15000]
if len(thin):
    print("Months with too little data (re-run Step 2 to fill them):", ", ".join(thin.index))
else:
    print("Every month looks complete.")


Cost per position: $0.16 (Exness median spread in New York hours); stress test: $0.32
Gold minutes loaded: 968,409  from 2024-01-01 to 2026-10-13
Every month looks complete.


**Step 3.** Load the trading logic (prints nothing). Same rules as the bake-off; only the moment the limit order goes in changes.

In [3]:
DISPLACEMENT = 1.5
BUFFER = 0.50


def swing_points(b, side):
    s = b["low"] if side == "low" else b["high"]
    if side == "low":
        return (s < s.shift(1)) & (s < s.shift(2)) & (s < s.shift(-1)) & (s < s.shift(-2))
    return (s > s.shift(1)) & (s > s.shift(2)) & (s > s.shift(-1)) & (s > s.shift(-2))


def find_setup2(b, lh, ll, mode):
    """Same rules as the bake-off's find_setup, with three timings for the limit order:
    A = as tested before; B = placed when the break bar closes (gap must exist by then);
    C = placed when the bar after the break closes (what the bot will do)."""
    body = (b["close"] - b["open"]).abs()
    avg_body = body.rolling(20, min_periods=5).median().shift(1)
    ny = b[(b.index.hour >= 13) & (b.index.hour < 17)]
    for t in ny.index:
        i = b.index.get_loc(t)
        bar = b.iloc[i]
        for d, swept in ((-1, bar["high"] > lh), (1, bar["low"] < ll)):
            if not swept:
                continue
            prior = b.iloc[max(0, i - 40):i]
            sw = swing_points(b.iloc[max(0, i - 42):i + 1], "low" if d == -1 else "high")
            sw = sw.reindex(prior.index).fillna(False).astype(bool)
            if not sw.any():
                continue
            level = prior.loc[sw[sw].index[-1], "low" if d == -1 else "high"]
            extreme = bar["high"] if d == -1 else bar["low"]
            for j in range(i + 1, min(i + 30, len(b))):
                x = b.iloc[j]
                extreme = max(extreme, x["high"]) if d == -1 else min(extreme, x["low"])
                broke = x["close"] < level if d == -1 else x["close"] > level
                ab = avg_body.iloc[j]
                strong = body.iloc[j] >= DISPLACEMENT * (ab if ab and not np.isnan(ab) else np.inf)
                if broke and strong:
                    kmax = j if mode == "B" else j + 1
                    for k in range(kmax, i, -1):
                        if k >= len(b) or k - 2 < 0:
                            continue
                        a, c = b.iloc[k - 2], b.iloc[k]
                        hit = (d == -1 and a["low"] > c["high"]) or (d == 1 and a["high"] < c["low"])
                        if not hit:
                            continue
                        entry = c["high"] if d == -1 else c["low"]
                        stop = extreme + BUFFER if d == -1 else extreme - BUFFER
                        ready = b.index[min(k + 1, len(b) - 1)]
                        det = b.index[j] + pd.Timedelta(minutes=5)
                        if mode == "B":
                            ready, watch = max(ready, det), det
                        elif mode == "C":
                            ready, watch = max(ready, det + pd.Timedelta(minutes=5)), det
                        else:
                            watch = ready
                        return dict(dir=d, entry=entry, stop=stop, ready=ready, watch=watch, k=k, j=j)
                    return None
            return None
    return None


def find_fill2(s, d1):
    d, entry, stop = s["dir"], s["entry"], s["stop"]
    # a stop touch between the break and the order going in cancels the setup
    pre = d1[(d1.index >= s["watch"]) & (d1.index < s["ready"])]
    if len(pre) and ((d == -1 and pre["high"].max() >= stop) or (d == 1 and pre["low"].min() <= stop)):
        return None
    after = d1[(d1.index >= s["ready"]) & (d1.index.hour < 21)]
    for t, x in after.iterrows():
        if (d == -1 and x["high"] >= stop) or (d == 1 and x["low"] <= stop):
            return None
        if (d == -1 and x["high"] >= entry) or (d == 1 and x["low"] <= entry):
            return None if t.hour >= 20 else t
    return None


def manage(s, adds=(), sizes=(1,), trail=False, overnight=False):
    """Net result in R (1R = first position's risk) and total size, for one setup."""
    d, entry, stop, risk = s["dir"], s["entry"], s["stop"], s["risk"]
    path = s["path"] if overnight else s["path"][s["path"].index < s["end_same"]]
    after = path[path.index > s["fill_t"]]
    hi, lo = after["high"].to_numpy(), after["low"].to_numpy()
    units, todo = [(entry, sizes[0])], list(adds)
    exit_price = path["close"].iloc[-1]
    for k in range(len(after)):
        if (d == 1 and lo[k] <= stop) or (d == -1 and hi[k] >= stop):
            exit_price = stop
            break
        best = (hi[k] - entry) if d == 1 else (entry - lo[k])
        while todo and best >= todo[0] * risk:
            level = entry + d * todo.pop(0) * risk
            units.append((level, sizes[min(len(units), len(sizes) - 1)]))
            stop = units[-2][0] if trail else entry
    gross = sum(sz * d * (exit_price - e) for e, sz in units)
    size = sum(sz for _, sz in units)
    return (gross - size * SPREAD) / risk, size


**Step 4.** Run the model three ways on 2024–2026 (a few minutes).

In [4]:
ADDS_TO_4R = dict(adds=(1, 2, 3, 4), sizes=(1,))
MODES = {"A": "Original test (order could start before the break was confirmed)",
         "B": "Order placed the moment the break bar closes",
         "C": "Order placed one bar after the break (what the bot will do)"}

days = sorted(set(m1.index.date))
by_day = {d: g for d, g in m1.groupby(m1.index.date)}
setups = {}
for mode in MODES:
    S = []
    for day, d1 in by_day.items():
        london = d1[(d1.index.hour >= 7) & (d1.index.hour < 12)]
        if len(london) < 200:
            continue
        b = d1.resample("5min").agg({"open": "first", "high": "max", "low": "min", "close": "last"}).dropna()
        s = find_setup2(b, london["high"].max(), london["low"].min(), mode)
        if not s:
            continue
        fill_t = find_fill2(s, d1)
        if fill_t is None:
            continue
        jx = days.index(day)
        next_day = days[jx + 1] if jx + 1 < len(days) else day
        end_next = pd.Timestamp(next_day).tz_localize("UTC") + pd.Timedelta(hours=21)
        end_same = pd.Timestamp(day).tz_localize("UTC") + pd.Timedelta(hours=21)
        S.append(dict(day=pd.Timestamp(day), year=day.year, dir=s["dir"], entry=s["entry"], stop=s["stop"],
                      risk=abs(s["entry"] - s["stop"]), fill_t=fill_t, ready=s["ready"], end_same=end_same,
                      gap_bar=s["k"] - s["j"], path=m1[(m1.index >= fill_t) & (m1.index < end_next)]))
    setups[mode] = S

rows = []
for mode, S in setups.items():
    for s in S:
        rows.append(dict(mode=mode, year=s["year"], hold=manage(s)[0], adds=manage(s, **ADDS_TO_4R)[0]))
R = pd.DataFrame(rows)


def tstat(x):
    return x.mean() / x.std() * np.sqrt(len(x))


print(f"Cost per position: ${SPREAD:.2f}\n")
for mode, label in MODES.items():
    g = R[R["mode"] == mode]
    print(f"{mode}. {label}")
    print(f"   trades {len(g)} | hold: {g.hold.mean():+.3f}R per trade (t = {tstat(g.hold):.2f})"
          f" | adds to +4R: {g.adds.mean():+.3f}R per trade (t = {tstat(g.adds):.2f})")
print("\n## Average R per trade by year, adds to +4R\n")
print(R.pivot_table(index="mode", columns="year", values="adds", aggfunc="mean").round(3).to_string())
late = sum(s["gap_bar"] < 0 for s in setups["A"])
print(f"\nIn the original test, {late} of {len(setups['A'])} trades used a gap that existed before the break,"
      " so the order could start before the break was confirmed.")


Cost per position: $0.16

A. Original test (order could start before the break was confirmed)
   trades 214 | hold: +0.193R per trade (t = 1.89) | adds to +4R: +0.546R per trade (t = 1.95)
B. Order placed the moment the break bar closes
   trades 136 | hold: +0.032R per trade (t = 0.27) | adds to +4R: +0.009R per trade (t = 0.04)
C. Order placed one bar after the break (what the bot will do)
   trades 207 | hold: +0.164R per trade (t = 1.57) | adds to +4R: +0.540R per trade (t = 1.88)

## Average R per trade by year, adds to +4R

year   2024   2025   2026
mode                     
A     0.104  0.862  0.926
B    -0.088 -0.481  0.763
C     0.129  0.874  0.850

In the original test, 41 of 214 trades used a gap that existed before the break, so the order could start before the break was confirmed.


**Step 5.** Robustness checks on the real-time version (C).

In [5]:
S = setups["C"]
orig = np.array([manage(s, **ADDS_TO_4R)[0] for s in S])
size = np.array([manage(s, **ADDS_TO_4R)[1] for s in S])
risk = np.array([s["risk"] for s in S])
print(f"Adds to +4R, normal spread ${SPREAD:.2f}:  {orig.mean():+.3f}R per trade")
print(f"Adds to +4R, double spread ${SPREAD_HIGH:.2f}:  {(orig - size * SPREAD / risk).mean():+.3f}R per trade")

flip = []
for s in S:
    s2 = dict(s, dir=-s["dir"], stop=2 * s["entry"] - s["stop"])
    flip.append(manage(s2, **ADDS_TO_4R)[0])
flip = np.array(flip)
rng = np.random.default_rng(1)
sims = np.array([np.where(rng.random(len(S)) < 0.5, orig, flip).mean() for _ in range(2000)])
print(f"Random direction beats the real one in {np.mean(sims >= orig.mean()) * 100:.1f}% of 2,000 shuffles (p = {np.mean(sims >= orig.mean()):.3f})")

d = np.array([s["dir"] for s in S])
print(f"Buys:  {(d == 1).sum()} trades, {orig[d == 1].mean():+.3f}R per trade")
print(f"Sells: {(d == -1).sum()} trades, {orig[d == -1].mean():+.3f}R per trade")

through = 0
for s in S:
    first = s["path"].iloc[0]
    if s["fill_t"] == s["ready"] and ((s["dir"] == -1 and first["open"] >= s["entry"]) or (s["dir"] == 1 and first["open"] <= s["entry"])):
        through += 1
print(f"Trades where price was already past the entry when the order went in: {through} of {len(S)}"
      " (the bot enters at market there, at an equal or better price)")


Adds to +4R, normal spread $0.16:  +0.540R per trade
Adds to +4R, double spread $0.32:  +0.513R per trade
Random direction beats the real one in 1.2% of 2,000 shuffles (p = 0.012)
Buys:  86 trades, +0.670R per trade
Sells: 121 trades, +0.448R per trade
Trades where price was already past the entry when the order went in: 15 of 207 (the bot enters at market there, at an equal or better price)


**Step 6.** Save the reference trade list to `My Trades/results/reference_trades.csv`.

In [6]:
ref = pd.DataFrame([dict(day=s["day"].date(), direction="buy" if s["dir"] == 1 else "sell",
                          entry=round(s["entry"], 3), stop=round(s["stop"], 3), risk=round(s["risk"], 3),
                          order_time_utc=s["ready"], fill_time_utc=s["fill_t"],
                          R_hold=round(manage(s)[0], 3), R_adds_to_4R=round(manage(s, **ADDS_TO_4R)[0], 3))
                     for s in setups["C"]])
path = os.path.join(OUT, "reference_trades.csv")
ref.to_csv(path, index=False)
print(f"Saved {len(ref)} trades to {path}\n")
print(ref.tail(10).to_string(index=False))


Saved 207 trades to /content/drive/MyDrive/My Trades /results/reference_trades.csv

       day direction    entry     stop   risk            order_time_utc             fill_time_utc  R_hold  R_adds_to_4R
2026-08-28      sell 4581.839 4632.630 50.791 2026-08-28 14:10:00+00:00 2026-08-28 15:00:00+00:00   2.473         4.419
2026-09-04       buy 4411.929 4364.704 47.225 2026-09-04 13:35:00+00:00 2026-09-04 13:35:00+00:00   0.383         0.383
2026-09-15      sell 4279.290 4301.115 21.825 2026-09-15 14:45:00+00:00 2026-09-15 14:45:00+00:00  -1.007        -1.007
2026-09-16      sell 4330.788 4361.038 30.250 2026-09-16 14:10:00+00:00 2026-09-16 14:10:00+00:00  -1.005        -1.005
2026-09-17      sell 4357.528 4380.516 22.988 2026-09-17 15:00:00+00:00 2026-09-17 15:05:00+00:00   0.674         0.674
2026-09-25       buy 4289.210 4253.728 35.482 2026-09-25 16:05:00+00:00 2026-09-25 16:14:00+00:00  -0.098        -0.098
2026-09-28       buy 4152.280 4135.866 16.414 2026-09-28 13:55:00+00:00 2026